# Governed loan policy walkthrough

This notebook follows policy text through Knowlytix GMS, then uses typed policy claims to route synthetic applications for human review. It also demonstrates action gates, integrity-protected tracing, evidence generation, and lineage.


In [34]:
from pathlib import Path
import sys
import pandas as pd

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
import loanguard as loan

applications = pd.read_csv(PROJECT / "synthetic_applications.csv")
applications.head()


,applicant_id,product,requested_amount,credit_score,annual_income,annual_debt,dti_percent,proposed_apr,marital_status,synthetic_outcome
0,A-1001,Personal Loan,12000,712,78000,21000,26.9,16.99,Married,EligibleForUnderwriting
1,A-1002,Personal Loan,28000,735,112000,30000,26.8,18.50,Single,RejectedByPolicyScreen
2,A-1003,Auto Loan,24500,625,64000,24000,37.5,12.99,Single,EligibleForUnderwriting
3,A-1004,Auto Loan,64000,701,135000,42000,31.1,17.99,Married,RejectedByPolicyScreen
4,A-1005,Home Equity Line,90000,724,165000,52000,31.5,9.75,Divorced,EligibleForUnderwriting


## 1. Load the Knowlytix graph and calibrated verifier

The store ingests `loan_policy.md` using deterministic regex extraction. The verifier thresholds are calibrated from the graph; this is a small educational corpus.


In [35]:
store = loan.build_or_load_store(rebuild=False, epochs=100)
verifier = loan.build_verifier(store)
print(f"GMS triples: {len(list(store.doc_graph.triples))}")
print(f"ENM entries: {store.stats()['enm_entries']}")


  GMS entities:  52
  GMS relations: 11
  GMS triples:   53
  Store loaded from N:\Solytics\Loan_Approval\loan_store
  Entities:  52
  Relations: 11
  Triples:   53
  ENM:       18
  Documents: 1
[1] loaded existing store from N:\Solytics\Loan_Approval\loan_store
  Calibrated geodesic      :   106 samples, 2-class, train accuracy=59.4%  τ=1.5194694393873216
  Calibrated tension       :   530 samples, 2-class, train accuracy=94.3%  τ=DEGENERATE (no usable two-cut separation)
[2] tau_plausibility=1.4921 (95% CI [1.2627, 1.6926]), tension_status=degenerate
GMS triples: 53
ENM entries: 18


## 2. Route applications using typed thresholds

Each check becomes a Knowlytix `ClaimType.THRESHOLD` claim against a typed ENM entry. Three product-policy checks determine whether an application can proceed to underwriting. Income-verification trigger is recorded separately. A human decision remains mandatory in every case.


In [36]:
screened = pd.DataFrame([
    loan.screen_application(store, verifier, row)
    for row in applications.to_dict(orient="records")
])
view = applications.merge(screened, on=["applicant_id", "product"], suffixes=("_sample", ""))
view[["applicant_id", "product", "requested_amount", "credit_score",
      "routing", "income_verification_required", "human_decision_required"]]


,applicant_id,product,requested_amount,credit_score,routing,income_verification_required,human_decision_required
0,A-1001,Personal Loan,12000,712,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,True
1,A-1002,Personal Loan,28000,735,MANUAL_REVIEW,True,True
2,A-1003,Auto Loan,24500,625,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,True
3,A-1004,Auto Loan,64000,701,MANUAL_REVIEW,True,True
4,A-1005,Home Equity Line,90000,724,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,True
5,A-1006,Home Equity Line,45000,681,MANUAL_REVIEW,True,True
6,A-1007,Student Refinance Loan,52000,694,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,True
7,A-1008,Student Refinance Loan,32000,658,MANUAL_REVIEW,True,True
8,A-1009,Small Business Line,76000,708,POLICY_ELIGIBLE_FOR_UNDERWRITING,True,True
9,A-1010,Small Business Line,115000,741,MANUAL_REVIEW,True,True


## Three explicit policy-screen examples

These examples show **PASS**, **REJECTED**, and **REVIEW** using the policy ENM checks. REJECTED means at least one checked policy limit failed, so the demo marks the application as rejected by its pre-screen. It is not a final loan decision; a human Loan Officer remains the approval authority. A PASS only qualifies for underwriting; a human decision is still required. REVIEW means a required value or policy limit is unavailable, so the check cannot be completed.


In [37]:
examples = [
    {
        "example": "PASS example",
        "applicant_id": "DEMO-PASS",
        "product": "Personal Loan",
        "requested_amount": 12000,
        "credit_score": 712,
        "proposed_apr": 16.99,
    },
    {
        "example": "REJECTED example",
        "applicant_id": "DEMO-REJECTED",
        "product": "Personal Loan",
        "requested_amount": 28000,  # Above the policy maximum of $25,000.
        "credit_score": 735,
        "proposed_apr": 18.50,
    },
    {
        "example": "REVIEW example",
        "applicant_id": "DEMO-REVIEW",
        "product": "Personal Loan",
        "requested_amount": 15000,
        "credit_score": 700,
        "proposed_apr": None,  # Without APR, one required threshold is unknown.
    },
]
example_results = [
    {**example, **loan.screen_application(store, verifier, example)}
    for example in examples
]
example_rows = []
for result in example_results:
    checks = result["checks"]
    example_rows.append({
        "example": result["example"],
        "screening_status": result["screening_status"],
        "screening_result": result["screening_result"],
        "amount_check": checks["requested_amount_within_policy_maximum"]["status"],
        "score_check": checks["credit_score_meets_policy_minimum"]["status"],
        "APR_check": checks["proposed_apr_within_policy_maximum"]["status"],
        "income_verification_required": result["income_verification_required"],
        "routing": result["routing"],
        "human_decision_required": result["human_decision_required"],
    })
pd.DataFrame(example_rows)


print("\nKnowlytix policy evidence by example:")
for result in example_results:
    print(f"\n{result['example']} — {result['screening_result']}")
    for name, check in result["checks"].items():
        print(f"  {name}: {check['status']} — {check['evidence']}")



Knowlytix policy evidence by example:

PASS example — PASSED_TO_UNDERWRITING
  requested_amount_within_policy_maximum: PASS — lookup_enm(loan_product_limits, Personal Loan/Loan Officer/Maximum Amount) = 25000.000000 ge 12000.000000 -> PASS (margin = +13000.000000)
  credit_score_meets_policy_minimum: PASS — lookup_enm(loan_product_limits, Personal Loan/Loan Officer/Minimum Credit Score) = 640.000000 le 712.000000 -> PASS (margin = +72.000000)
  proposed_apr_within_policy_maximum: PASS — lookup_enm(loan_product_limits, Personal Loan/Loan Officer/Maximum APR) = 24.990000 ge 16.990000 -> PASS (margin = +8.000000)
  income_verification_trigger_exceeded: PASS — lookup_enm(processing_standards, Income Verification Required Above Amount) = 10000.000000 lt 12000.000000 -> PASS (margin = +2000.000000)

REJECTED example — REJECTED_FROM_AUTO_PASS
  requested_amount_within_policy_maximum: FAIL — lookup_enm(loan_product_limits, Personal Loan/Loan Officer/Maximum Amount) = 25000.000000 ge 28000.000

In [38]:
# Labels are synthetic examples for a consistency check, not ground truth.
expected = view["synthetic_outcome"].map({"EligibleForUnderwriting": "POLICY_ELIGIBLE_FOR_UNDERWRITING", "RejectedByPolicyScreen": "PRE_SCREEN_REJECTED", "ManualReview": "MANUAL_REVIEW"})
view["label_match"] = view["routing"] == expected
print(f"Illustrative route agreement: {view['label_match'].sum()}/{len(view)}")
view["routing"].value_counts()


Illustrative route agreement: 5/12


routing
MANUAL_REVIEW                       7
POLICY_ELIGIBLE_FOR_UNDERWRITING    5
Name: count, dtype: int64

## 3. Inspect a policy entry and claim verdicts

The repaired ENM lookup handles Knowlytix `ENMKey` objects and reads values through `lookup_enm`, rather than assuming tuple-shaped keys.


In [39]:
amount_key = loan.find_enm(store, "Personal Loan", "Maximum Amount")
print("ENM key:", amount_key)
print("Policy value:", loan.enm_value(store, amount_key))
claim_rows = loan.build_claims(store)
claim_verdicts = verifier.verify_all([claim for _, claim, _ in claim_rows])
for (label, _, expected), verdict in zip(claim_rows, claim_verdicts.claim_verdicts):
    print({"claim": label, "expected": expected, "observed": verdict.passed,
           "severity": verdict.severity, "evidence": verdict.evidence})


ENM key: ENMKey(type='loan_product_limits', id='Personal Loan/Loan Officer/Maximum Amount', timestamp=1791187790.91427)
Policy value: 25000.0
{'claim': 'approval is guaranteed', 'expected': False, 'observed': False, 'severity': 'error', 'evidence': "Entity not found: tail 'Permitted'"}
{'claim': 'Senior Underwriter approves a personal loan', 'expected': False, 'observed': False, 'severity': 'error', 'evidence': 'score_triple(personal loan, has_approval_authority, senior underwriter) = 1.6158 >= 1.4921228142219538 (no graph match)'}
{'claim': '(control) Loan Officer approves an auto loan', 'expected': True, 'observed': True, 'severity': 'info', 'evidence': '(auto loan, has_approval_authority, loan officer) verified via query_triples (exact graph match), score_triple=1.2955'}
{'claim': 'Policy maximum personal-loan amount is $25,000', 'expected': True, 'observed': True, 'severity': 'info', 'evidence': 'lookup_enm(loan_product_limits, Personal Loan/Loan Officer/Maximum Amount) = 25000.000

## 4. Apply declarative action policy

`PolicyEngine` checks assistant actions against explicit allow, require, and block rules. These checks govern proposed actions; they do not execute loan operations.


In [40]:
engine = loan.build_policy_engine(store)
action_checks = []
for action, args in loan.ACTION_PROBES:
    result = engine.check(action=action, args=args, context="lending_assistant")
    action_checks.append({"action": action, "allowed": result.allowed,
                          "blocking_rules": result.blocking_rules, "reason": result.reason})
pd.DataFrame(action_checks)


,action,allowed,blocking_rules,reason
0,lookup_product_terms,True,[],Explicitly allowed by policy rule
1,quote_rate,False,"[LG-002, gms_graph:quote_rate->requires->apr_d...",Rule LG-002: 'quote_rate' requires 'apr_disclo...
2,issue_loan_offer,False,"[LG-003, gms_graph:issue_loan_offer->requires-...",Rule LG-003: 'issue_loan_offer' requires 'huma...
3,waive_income_verification,False,"[LG-004, gms_graph:waive_income_verification->...",Rule LG-004: 'waive_income_verification' block...


## 5. Trace and verify integrity

The command-line workflow records verification and action decisions, seals the audit packets, checks the chain, then demonstrates detection of an edited packet. A hash chain detects edits; an external last-seen step is needed to detect tail truncation.


In [41]:
import copy
import importlib
import time

# Refresh the project helper in case this Jupyter kernel imported an older copy.
loan = importlib.reload(loan)
required_audit_helpers = (
    "load_audit_signing_key", "sign_audit_manifest", "verify_audit_manifest",
)
missing_helpers = [name for name in required_audit_helpers if not hasattr(loan, name)]
if missing_helpers:
    raise RuntimeError(
        f"The loaded loanguard.py is missing {missing_helpers}; verify the project path: {loan.__file__}"
    )
from knowlytix.harness.governance.tracing_hardening import (
    seal_audit_trail, verify_audit_chain, chain_head_hash,
    compute_packet_content_hash,
)
from knowlytix.harness.governance.tracing import GovernanceTracer

tracer = GovernanceTracer()
tracer.start_run("notebook_demo", "Loan policy verification")
for step, (_, claim, _) in enumerate(claim_rows):
    started = time.perf_counter()
    v = verifier.verify_all([claim]).claim_verdicts[0]
    elapsed_ms = (time.perf_counter() - started) * 1000.0
    packet = tracer.trace_tool_call(run_id="notebook_demo", step_num=step,
        tool_name="verify_claim", args={"claim": claim.text},
        result_summary=v.evidence, policy_decision="allowed" if v.passed else "blocked",
        gms_scores={"pass_confidence": float(v.score),
                    **({"geodesic_distance": float(v.raw_signal)}
                       if v.raw_signal is not None and v.gms_method == "score_triple" else {})},
        gate_results={"verification": "PASS" if v.passed else "BLOCK"},
        execution_time_ms=elapsed_ms,
        state_before="DRAFT", state_after="CHECKED")
    packet.verification = {
        "passed": bool(v.passed), "severity": v.severity,
        "method": v.gms_method, "raw_signal": v.raw_signal,
        "evidence": str(v.evidence)[:500],
    }
tracer.end_run("notebook_demo")
packets = tracer.get_audit_trail("notebook_demo")
seal_audit_trail(packets)
print({"packets": len(packets), "head": chain_head_hash(packets),
       "integrity_errors": verify_audit_chain(packets)})

# The plain hash chain can be recomputed. The separate signed manifest anchors
# the full packet sequence and final step when its HMAC key is kept private.
key, _ = loan.load_audit_signing_key()
manifest = loan.sign_audit_manifest(packets, "notebook_demo", key)
forged = copy.deepcopy(packets)
if len(forged) > 1:
    forged[0].result_summary += " [tamper probe]"
    forged[0].content_hash = compute_packet_content_hash(forged[0])
    forged[1].prev_hash = forged[0].content_hash
print({"recomputed_hash_chain_breaks": len(verify_audit_chain(forged)),
       "forged_manifest_valid": loan.verify_audit_manifest(forged, manifest, key),
       "tail_truncated_manifest_valid": loan.verify_audit_manifest(packets[:-1], manifest, key)})


{'packets': 8, 'head': 'f1677a145eb28b86f3e000d62c2880089e7e075c74f46cc801f79ff39223e319', 'integrity_errors': []}
{'recomputed_hash_chain_breaks': 0, 'forged_manifest_valid': False, 'tail_truncated_manifest_valid': False}


## 6. Demo evidence summary and lineage
This section reports observed checks and provenance for this synthetic run. It is not a compliance assessment, certification, or independent validation.


In [42]:
summary = loan.build_demo_evidence_summary(
    decision=loan.release_decision(claim_verdicts),
    claim_passed=sum(1 for v in claim_verdicts.claim_verdicts if v.passed),
    claim_total=len(claim_verdicts.claim_verdicts),
    action_allowed=sum(1 for x in action_checks if x['allowed']),
    action_blocked=sum(1 for x in action_checks if not x['allowed']),
    route_matches=int(view['label_match'].sum()), application_count=len(view),
    audit_entries=len(packets),
    audit_chain_valid=not loan.verify_audit_chain(packets),
    audit_manifest_valid=loan.verify_audit_manifest(packets, manifest, key),
    tension_status=verifier.calibrated_thresholds.tension_status,
)
summary_path = loan.OUT_DIR / 'notebook_demo_evidence_summary.md'
summary_path.parent.mkdir(exist_ok=True)
summary_path.write_text(summary, encoding='utf-8')
stale_report = loan.OUT_DIR / 'compliance_package.md'
if stale_report.exists(): stale_report.unlink()
print(summary)
print('Wrote:', summary_path)
from knowlytix.harness.governance.compliance import ProvenanceRegistry
registry = ProvenanceRegistry()
registry.register(name='loan_policy.md', artifact_type='dataset', source='demo', metadata={'synthetic': True})
registry.register(name='loan_store', artifact_type='model', source='Knowlytix GMS ingest', metadata={'mode': 'regex'}, parent='loan_policy.md')
registry.register(name='screened_applications', artifact_type='output', source='ClaimVerifier', metadata={'rows': len(view)}, parent='loan_store')
registry.get_lineage('screened_applications')


# LoanGuard demo evidence summary ? not a compliance assessment

This report describes one run of a synthetic educational example. It is not a certification, legal opinion, production model evaluation, or loan decision.

## Run observations

- Assistant-reply claim checks: 4/8 passed (50%). The test cohort includes deliberately false claims; this rate is not model accuracy or an Effective Challenge pass rate.
- Final reply gate: `BLOCK_AND_ESCALATE_TO_HUMAN`.
- Synthetic route-label agreement: 5/12. The labels and routing rules derive from the same demo rules, so this is a consistency check, not independent validation.
- Action probes: 1 allowed and 3 blocked among four scripted probes; these are examples, not real-world policy violations.
- Audit chain valid: `True`; signed manifest valid: `True`; packets: 8.
- Tension calibration: `degenerate`. Contradiction/tension checks are unavailable when calibration is degenerate.

## Scope and limits

Applicant data and policy are synthetic. G

[{'name': 'loan_policy.md',
  'type': 'dataset',
  'source': 'demo',
  'metadata': {'synthetic': True},
  'parent': None,
  'timestamp': '2026-10-05T08:09:55.884965+00:00',
  'fingerprint': '00df3c6b62926a46'},
 {'name': 'loan_store',
  'type': 'model',
  'source': 'Knowlytix GMS ingest',
  'metadata': {'mode': 'regex'},
  'parent': 'loan_policy.md',
  'timestamp': '2026-10-05T08:09:55.884965+00:00',
  'fingerprint': '18a2460bf746fe7e'},
 {'name': 'screened_applications',
  'type': 'output',
  'source': 'ClaimVerifier',
  'metadata': {'rows': 12},
  'parent': 'loan_store',
  'timestamp': '2026-10-05T08:09:55.884965+00:00',
  'fingerprint': 'fda674017685c3ce'}]

## Summary

The workflow grounds threshold checks in policy values stored in Knowlytix and records why each check passed or failed. Small synthetic examples are useful for exercising the controls, not for estimating lending risk or approving applicants.
